# Epistemic Event Replay

This notebook replays optimization runs as a timeline and lineage tree, enabling
insight into how different arms reason about instruction changes.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Add parent to path
sys.path.insert(0, str(Path.cwd().parent))

from events import EventStorage, EventAnalyzer
from events.schema import LakatosLabel

%matplotlib inline
sns.set_style('whitegrid')

## Load Events

In [ ]:
# Initialize storage
results_dir = Path("../results")
storage = EventStorage(results_dir)

# List available runs
event_files = list(storage.events_dir.glob("events_*.jsonl"))
print(f"Found {len(event_files)} event files:")
for f in event_files[:10]:
    print(f"  {f.name}")

In [ ]:
# Load events for a specific run
# Replace with actual run_id
run_id = "run_001"

events = storage.get_run_events(run_id)
print(f"Loaded {len(events)} events for run {run_id}")

## Timeline View

In [ ]:
# Extract timeline
analyzer = EventAnalyzer(events)
timeline = analyzer.extract_timeline()

# Convert to DataFrame
df_timeline = pd.DataFrame(timeline)

# Filter to optimization steps
df_steps = df_timeline[df_timeline['event_type'] == 'optimization_step'].copy()

print(f"Timeline: {len(df_steps)} optimization steps")
df_steps.head()

In [ ]:
# Plot timeline
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Accepted vs rejected over time
ax = axes[0]
accepted = df_steps['accepted'].cumsum()
rejected = (~df_steps['accepted']).cumsum()

ax.plot(df_steps['iteration'], accepted, label='Accepted', marker='o')
ax.plot(df_steps['iteration'], rejected, label='Rejected', marker='x')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cumulative Count')
ax.set_title('Accepted vs Rejected Candidates Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Bottom: Lakatos labels over time
ax = axes[1]
label_colors = {
    'progressive': 'green',
    'degenerating': 'red',
    'neutral': 'gray',
    'unknown': 'lightgray',
}

for label, color in label_colors.items():
    mask = df_steps['lakatos_label'] == label
    if mask.any():
        subset = df_steps[mask]
        ax.scatter(subset['iteration'], subset['cost'], 
                  label=label, color=color, alpha=0.6, s=100)

ax.set_xlabel('Iteration')
ax.set_ylabel('Cost per Step')
ax.set_title('Lakatos Labels Over Time (Color = Label, Size = Cost)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Lineage Tree

In [ ]:
# Build lineage tree
tree = analyzer.build_lineage_tree()

print("Lineage tree:")
for parent, children in tree.items():
    print(f"  {parent} -> {len(children)} children")
    for child in children[:3]:
        print(f"    - {child}")

In [ ]:
# Visualize lineage as text tree
def print_tree(node, tree, depth=0, max_depth=5):
    if depth > max_depth:
        return
    
    indent = "  " * depth
    print(f"{indent}{node}")
    
    if node in tree:
        for child in tree[node]:
            print_tree(child, tree, depth + 1, max_depth)

print("\nLineage tree (from root):")
print_tree("root", tree)

## Lakatos Analysis

In [ ]:
# Count by Lakatos label
lakatos_counts = analyzer.count_by_lakatos_label()

print("\nLakatos label distribution:")
for label, count in lakatos_counts.items():
    print(f"  {label.value}: {count}")

# Progressive ratio
progressive_ratio = analyzer.get_progressive_ratio()
if progressive_ratio >= 0:
    print(f"\nProgressive ratio: {progressive_ratio:.2%}")

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
labels = [l.value for l in lakatos_counts.keys()]
counts = list(lakatos_counts.values())
colors = [label_colors.get(l, 'gray') for l in labels]

ax.bar(labels, counts, color=colors, alpha=0.7)
ax.set_xlabel('Lakatos Label')
ax.set_ylabel('Count')
ax.set_title('Distribution of Lakatos Labels')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Cost Breakdown

In [ ]:
# Compute cost breakdown
cost_breakdown = analyzer.compute_cost_breakdown()

print("\nCost breakdown:")
print(f"  Total calls: {cost_breakdown['total_calls']}")
print(f"  Total cost: ${cost_breakdown['total_cost']:.4f}")

print("\nBy Lakatos label:")
for label, data in cost_breakdown['by_label'].items():
    print(f"  {label.value}:")
    print(f"    Calls: {data['calls']}")
    print(f"    Cost: ${data['cost']:.4f}")

## Cross-Arm Comparison

Compare how different arms approach the same example.

In [ ]:
# Load events for multiple arms on same benchmark
arms = ['A', 'B', 'C']  # Example

arm_events = {}
for arm in arms:
    # Load events for this arm
    # In practice, would filter by run_id and arm_name
    arm_events[arm] = []  # Placeholder

print("\nCross-arm comparison:")
for arm, events in arm_events.items():
    if events:
        analyzer = EventAnalyzer(events)
        ratio = analyzer.get_progressive_ratio()
        print(f"  Arm {arm}: {len(events)} steps, progressive ratio {ratio:.2%}")

## Refutation Patterns (Popperian Arms)

In [ ]:
# Extract refutation patterns
refutations = analyzer.get_refutation_patterns()

if refutations:
    df_ref = pd.DataFrame(refutations)
    
    print(f"\nFound {len(refutations)} refutations")
    
    # Plot severity distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(df_ref['severity'], bins=20, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Severity')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Refutation Severity')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Show high-severity refutations
    print("\nHigh-severity refutations (>0.8):")
    high_severity = df_ref[df_ref['severity'] > 0.8]
    for _, row in high_severity.iterrows():
        print(f"  Step {row['step_id']}: {row['claim'][:80]}...")
        print(f"    Severity: {row['severity']:.2f}, Counterexamples: {row['n_counterexamples']}")
else:
    print("\nNo refutations found (not a Popperian arm?)")

## Clause Credence Evolution (Bayesian Arms)

In [ ]:
# Track clause credence evolution
evolution = analyzer.get_clause_credence_evolution()

if evolution:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for clause_id, credences in evolution.items():
        ax.plot(credences, label=f"Clause {clause_id}", marker='o')
    
    ax.set_xlabel('Step')
    ax.set_ylabel('Credence')
    ax.set_title('Clause Credence Evolution Over Time')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("\nNo clause credences found (not a Bayesian arm?)")

## Export Summary

In [ ]:
# Export summary statistics
summary = {
    "run_id": run_id,
    "total_steps": len(df_steps),
    "accepted": int(df_steps['accepted'].sum()),
    "rejected": int((~df_steps['accepted']).sum()),
    "lakatos_distribution": {k.value: v for k, v in lakatos_counts.items()},
    "progressive_ratio": progressive_ratio,
    "cost_breakdown": cost_breakdown,
}

# Save
summary_path = results_dir / f"event_summary_{run_id}.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nSummary saved to {summary_path}")